In [0]:
%python
# ABS Building Approvals Silver transformation (Delta)
from pyspark.sql import functions as F
import json

dbutils.widgets.text("ingestion_date", "")
dbutils.widgets.text("run_id", "")

INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()

bronze_path = f"s3://johnq-data-lake-dev/abs-building-approvals/bronze/ingestion_date={INGESTION_DATE}/"
silver_path = "s3://johnq-data-lake-dev/abs-building-approvals/silver/"


In [0]:
%python
# Fix Parquet physical type mismatches across files (e.g., INT64 vs DOUBLE)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

df = spark.read.parquet(bronze_path)


In [0]:
%python
# Split "code: label" into two columns
def split_code_label(colname: str):
    parts = F.split(F.col(colname), ":")
    code = F.trim(parts.getItem(0))
    label = F.when(F.size(parts) >= 2, F.trim(parts.getItem(1))).otherwise(F.lit(None))
    return code, label

dataflow_code, dataflow_label = split_code_label("dataflow")
measure_code, measure_label = split_code_label("measure")
sector_code, sector_label = split_code_label("sector")
work_type_code, work_type_label = split_code_label("work_type")
building_type_code, building_type_label = split_code_label("building_type")
region_type_code, region_type_label = split_code_label("region_type")
region_code, region_label = split_code_label("region")
frequency_code, frequency_label = split_code_label("frequency")
unit_measure_code, unit_measure_label = split_code_label("unit_measure")
unit_mult_code, unit_mult_label = split_code_label("unit_mult")

In [0]:
%python
# Create silver dataframe
df_silver = (
    df.select(
        "time_period",
        "obs_value",
        "obs_status",
        "obs_comment",

        dataflow_code.alias("dataflow_code"),
        dataflow_label.alias("dataflow_label"),

        measure_code.alias("measure_code"),
        measure_label.alias("measure_label"),

        sector_code.alias("sector_code"),
        sector_label.alias("sector_label"),

        work_type_code.alias("work_type_code"),
        work_type_label.alias("work_type_label"),

        building_type_code.alias("building_type_code"),
        building_type_label.alias("building_type_label"),

        region_type_code.alias("region_type_code"),
        region_type_label.alias("region_type_label"),

        region_code.alias("region_code"),
        region_label.alias("region_label"),

        frequency_code.alias("frequency_code"),
        frequency_label.alias("frequency_label"),

        unit_measure_code.alias("unit_measure_code"),
        unit_measure_label.alias("unit_measure_label"),

        unit_mult_code.alias("unit_mult_code"),
        unit_mult_label.alias("unit_mult_label"),

        "region_file",
        "ingestion_date",
    )
    # Normalise types and add partition column
    .withColumn("obs_value", F.col("obs_value").cast("double"))
    .withColumn("ingestion_date", F.lit(INGESTION_DATE))
)

df_silver.limit(5).show(truncate=False)



In [0]:
%python
# Overwrite only this partition, not the whole table
(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
    .partitionBy("ingestion_date")
    .save(silver_path)
)

print("✅ Wrote silver Delta to:", silver_path)
spark.read.format("delta").load(silver_path).limit(5).show(truncate=False)